# ASG-WM / FaithCast — **Eval All (ours)** → paper assets

Evaluates **our** model (ASG-WM) from trained checkpoints and produces **`paper_assets.zip`** containing every value (`*_results.json`, `tables/*.tex`) and image (`figures/*.pdf` + `*.png`) the paper needs.

**Pipeline:** skill (A/B/F) → faithfulness (C) → figures + qualitative gallery → zip.

Run `train.ipynb` first (or unzip `train_results.zip` into `artifacts/`) so the checkpoints in `artifacts/ckpt` exist. Baselines stay `[TBR]` until trained — this notebook evaluates ours only.

In [ ]:
# --- 1. Locate / clone the repo ----------------------------------------------
# On Colab: set ASGWM_REPO_URL to your git remote, this cell clones it.
# Locally: just run this notebook from the repo root (the cell is a no-op).
import os, sys, subprocess, pathlib

REPO_URL = os.environ.get("ASGWM_REPO_URL", "")          # e.g. https://github.com/<you>/<repo>.git
REPO_DIR = "Forecasting-Through-Meteorological-Reasoning"

if not pathlib.Path("src/asgwm").exists():
    if not pathlib.Path(REPO_DIR, "src", "asgwm").exists():
        assert REPO_URL, "Set ASGWM_REPO_URL (Colab) or launch this notebook from the repo root."
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

assert pathlib.Path("src/asgwm").exists(), f"Run from repo root; cwd={os.getcwd()}"
print("repo root:", os.getcwd())


In [ ]:
# --- 2. Install dependencies --------------------------------------------------
# Core package + pinned requirements. Real-data extras are per-dataset (uncomment).
get_ipython().system('pip -q install -e ./src')
get_ipython().system('pip -q install -r src/requirements.txt')

# Real-data backends (only needed when DATASET != "synthetic"):
#   SEVIR  : pip -q install s3fs h5py
#   MRMS   : pip -q install boto3 cfgrib eccodes xarray
#   NEXRAD : pip -q install arm-pyart boto3


In [ ]:
# --- 3. Eval configuration ----------------------------------------------------
# Evaluate OUR model (ASG-WM) and produce every value + image the paper needs.
# In-distribution = SEVIR test; out-of-distribution generalization = NEXRAD, MRMS.
import subprocess, sys, itertools, os

DATASETS  = ["synthetic"]    # e.g. ["sevir", "nexrad", "mrms"] for the full paper run
N_EVAL    = 32               # eval events per dataset (raise for final numbers)
CFG       = "src/configs/default.yaml"

def ov(**kw):
    return list(itertools.chain.from_iterable(["--override", f"{k}={v}"] for k, v in kw.items()))

def run(script, results_dir, dataset, extra=None):
    common = ov(**{
        "data.dataset": dataset,
        "eval.n_eval_events": N_EVAL,
        "paths.results": results_dir,
    })
    cmd = [sys.executable, f"src/scripts/{script}", "--config", CFG] + common + (extra or [])
    print(">>", " ".join(cmd)); sys.stdout.flush()
    subprocess.run(cmd, check=True)


In [ ]:
# --- 4. Skill + faithfulness + figures, per dataset --------------------------
# Each dataset writes to its own results dir so OOD runs never clobber the main tables.
RESULT_DIRS = {}
for ds in DATASETS:
    rdir = f"artifacts/results_{ds}"
    RESULT_DIRS[ds] = rdir
    print(f"\n===== evaluating ASG-WM on: {ds}  ->  {rdir} =====")
    run("40_eval_skill.py",        rdir, ds)              # CSI/HSS/POD/FAR + tables/skill.tex
    run("41_eval_faithfulness.py", rdir, ds)             # C-i..C-iv faithfulness suite
    run("42_make_figures.py",      rdir, ds, ["--gallery"])  # PDF+PNG figures + qualitative gallery


In [ ]:
# --- 5. Bundle paper assets (values + images) -> paper_assets.zip ------------
import glob, os, zipfile

OUT = "paper_assets.zip"
with zipfile.ZipFile(OUT, "w", zipfile.ZIP_DEFLATED) as z:
    for ds, rdir in RESULT_DIRS.items():
        for p in glob.glob(os.path.join(rdir, "**", "*"), recursive=True):
            if os.path.isfile(p) and p.lower().endswith((".json", ".tex", ".pdf", ".png", ".csv")):
                z.write(p)
size_mb = os.path.getsize(OUT) / 1e6 if os.path.exists(OUT) else 0.0
print(f"wrote {OUT}  ({size_mb:.1f} MB)")
print("contains: *_results.json (values), tables/*.tex, figures/*.pdf+*.png  for every dataset")


**Done.** Download `paper_assets.zip` — drop the `figures/*.pdf` and `tables/*.tex` straight into `paper/`.